# 05 — Posterior predictive: forecasting with uncertainty

Notebook 04 showed that `BayesianVECM.fit` works — the model samples, the $\beta$-identification pin holds, and the posteriors recover the true parameters on synthetic data. But a model you can only fit is only half the job. The other half is **forecasting**: given a fitted posterior, what does the model say will happen next?

This notebook walks through `BayesianVECM.sample_posterior_predictive(steps)`, the method that was still a `NotImplementedError` stub in notebook 04 and is now live.

The core idea — **uncertainty propagates through the recursion** — is the headline teaching moment. At each step ahead we draw a fresh innovation $\varepsilon_{T+h} \sim \mathcal{N}(0, \Sigma)$ *for every posterior draw of $\Sigma$*. Those innovations compound: a step-1 shock shifts the level, which feeds into the error-correction term at step 2, and so on. The result is a forecast distribution that widens sensibly with the horizon rather than collapsing to a point estimate.

**Setup:** we use the same bivariate cointegrated DGP as notebook 04 — $\beta = (1, -0.5)^{\top}$, $\alpha = (-0.4, 0.2)^{\top}$ — but generate 100 observations instead of 300, hold out the last 20 as a test set, and ask the model to forecast those 20 steps. The fan chart at the end compares the posterior forecast distribution against the actuals.

In [ ]:
from __future__ import annotations

import matplotlib.pyplot as plt
import numpy as np

from bayesian_vecm import BayesianVECM

np.set_printoptions(precision=3, suppress=True)

In [ ]:
# ---------------------------------------------------------------------------
# Sampling configuration
# ---------------------------------------------------------------------------
# Set FAST_SAMPLING = True for quick execution (CI / first read-through).
# Set FAST_SAMPLING = False for publication-quality posteriors.
# ---------------------------------------------------------------------------
FAST_SAMPLING = True

if FAST_SAMPLING:
    DRAWS, TUNE, CHAINS = 200, 200, 2
else:
    DRAWS, TUNE, CHAINS = 1000, 1000, 4

print(f"Sampling config: draws={DRAWS}, tune={TUNE}, chains={CHAINS}")
if FAST_SAMPLING:
    print("(FAST_SAMPLING=True — posteriors are coarser; set to False for full quality)")

## 1. The DGP and the train/test split

Same DGP as notebook 04 — a bivariate VECM with no short-run dynamics:

$$\Delta y_t = \alpha\,\beta^{\top} y_{t-1} + \varepsilon_t, \qquad \varepsilon_t \sim \mathcal{N}(0, \sigma^2 I_2),$$

with $\beta = (1, -0.5)^{\top}$, $\alpha = (-0.4, 0.2)^{\top}$, and $\sigma = 0.5$.

We generate $T = 100$ observations, fit on the first 80, and forecast the last 20. The 20 held-out points are the actuals we'll overlay on the fan chart.

In [ ]:
def make_cointegrated_series(
    n_obs: int = 100,
    alpha_true: tuple[float, float] = (-0.4, 0.2),
    beta_true: tuple[float, float] = (1.0, -0.5),
    sigma: float = 0.5,
    seed: int = 0,
) -> np.ndarray:
    rng = np.random.default_rng(seed=seed)
    a0, a1 = alpha_true
    b0, b1 = beta_true
    y = np.zeros((n_obs, 2))
    y[0] = rng.normal(size=2)
    for t in range(1, n_obs):
        ec = b0 * y[t - 1, 0] + b1 * y[t - 1, 1]
        y[t, 0] = y[t - 1, 0] + a0 * ec + rng.normal(scale=sigma)
        y[t, 1] = y[t - 1, 1] + a1 * ec + rng.normal(scale=sigma)
    return y


# Generate full series, then split
T_TOTAL = 100
T_TRAIN = 80
T_TEST = T_TOTAL - T_TRAIN  # 20 held-out steps

endog_full = make_cointegrated_series(n_obs=T_TOTAL, seed=0)
endog_train = endog_full[:T_TRAIN]
endog_test = endog_full[T_TRAIN:]

print(f"Full series shape : {endog_full.shape}")
print(f"Training set      : rows 0..{T_TRAIN - 1}  ({T_TRAIN} obs)")
print(f"Test set (actuals): rows {T_TRAIN}..{T_TOTAL - 1} ({T_TEST} obs)")
print()
print("Training set — last 3 rows (these seed the forecast recursion):")
print(endog_train[-3:])

## 2. Fit the model on the training set

Nothing new here compared to notebook 04 — `fit` takes the training data and runs `pm.sample`. With only 80 observations and a bivariate model the sampler is fast; the posteriors won't be as tight as with 300 obs but they'll still recover the true parameters reasonably well.

We pass `cores=1` to sidestep the macOS multiprocessing issue noted in NOTES.md.

In [ ]:
model = BayesianVECM(k_ar_diff=1, coint_rank=1, deterministic="n")
model.fit(
    endog_train,
    draws=DRAWS,
    tune=TUNE,
    chains=CHAINS,
    random_seed=42,
    progressbar=False,
    cores=1,
)

print("Training complete.")
print(model.summary())

## 3. Forecast with `sample_posterior_predictive`

`sample_posterior_predictive(steps)` rolls the VECM recursion forward from the end of the training series:

$$
\Delta y_{T+h} = \alpha\,\beta^{\top} y_{T+h-1}
               + \Gamma_1\,\Delta y_{T+h-1}
               + \varepsilon_{T+h},
\qquad
y_{T+h} = y_{T+h-1} + \Delta y_{T+h},
$$

for $h = 1, \dots, \text{steps}$. The seed window — the last `k_ar_diff + 1 = 2` rows of `endog_train` — provides the last level and the last lagged difference needed to start the recursion.

For each posterior draw, a fresh $\varepsilon_{T+h}$ is drawn at every step from $\mathcal{N}(0, \Sigma_{(d)})$. The result is a *distribution* over forecast paths, not a point estimate.

The return type is an `xr.DataTree` with a `posterior_predictive` child node holding:
- `y` — forecast levels, shape `(chain, draw, steps, K)`.
- `delta_y` — forecast first differences, same shape.

In [ ]:
FORECAST_STEPS = T_TEST  # 20 — match the held-out window

pp = model.sample_posterior_predictive(steps=FORECAST_STEPS, random_seed=42)

# y_draws: shape (chain, draw, steps, K) -> reshape to (n_draws_total, steps, K)
y_draws = pp["posterior_predictive"]["y"].values
n_chains, n_draws, n_steps, n_vars = y_draws.shape
y_flat = y_draws.reshape(n_chains * n_draws, n_steps, n_vars)

print(f"Forecast output shape (chain, draw, steps, K): {y_draws.shape}")
print(f"Reshaped to (n_draws_total, steps, K)        : {y_flat.shape}")
print()
print("First 3 draws of y0 at step 1 (spot-check for non-zero spread):")
print(y_flat[:3, 0, 0])

## 4. Summarising the forecast distribution

With draws of each future level we can compute any posterior summary we like. For the fan chart we want:

- **Posterior median** — the central forecast path.
- **80% HDI** (10th–90th percentile) — a reasonably tight band.
- **94% HDI** (3rd–97th percentile) — a wider band showing the tails.

We use `np.quantile` over the draw axis.

In [ ]:
# Summaries over the draw dimension (axis 0 after flattening).
y_median = np.median(y_flat, axis=0)  # (steps, K)
y_q10 = np.quantile(y_flat, 0.10, axis=0)  # 80% HDI lower
y_q90 = np.quantile(y_flat, 0.90, axis=0)  # 80% HDI upper
y_q03 = np.quantile(y_flat, 0.03, axis=0)  # 94% HDI lower
y_q97 = np.quantile(y_flat, 0.97, axis=0)  # 94% HDI upper

# x-axis indices
x_train = np.arange(T_TRAIN)
x_forecast = np.arange(T_TRAIN, T_TRAIN + FORECAST_STEPS)

print("Forecast summary for y0 (first variable):")
print(f"{'step':>4}  {'median':>7}  {'80% lo':>7}  {'80% hi':>7}  {'actual':>7}")
for h in range(FORECAST_STEPS):
    print(
        f"{h + 1:>4}  {y_median[h, 0]:>7.3f}  {y_q10[h, 0]:>7.3f}  "
        f"{y_q90[h, 0]:>7.3f}  {endog_test[h, 0]:>7.3f}"
    )

## 5. The fan chart

A fan chart overlays the posterior forecast distribution against the held-out actuals. The shaded regions are the 80% and 94% HDI bands; the solid orange line is the posterior median; the dashed black line is what actually happened.

A few things to look for:

- **Do the bands contain the actuals most of the time?** For an 80% band you'd expect roughly 80% coverage — not perfect with 20 points, but gross failures are a red flag.
- **Do the bands widen with the horizon?** They should — uncertainty compounds through the recursion. A flat band would indicate innovations are being ignored.
- **Does the median track the general direction?** The error-correction mechanism pulls both series towards the cointegrating relation, so persistent deviations in the actuals should be loosely followed.

In [ ]:
CONTEXT_ROWS = 20  # training points to show for visual context
x_context = x_train[-CONTEXT_ROWS:]
var_labels = ["$y_0$", "$y_1$"]

fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)

for k, ax in enumerate(axes):
    ax.plot(
        x_context,
        endog_train[-CONTEXT_ROWS:, k],
        color="steelblue",
        linewidth=1.5,
        label="Training data",
    )
    ax.plot(
        x_forecast,
        endog_test[:, k],
        color="black",
        linewidth=1.5,
        linestyle="--",
        label="Held-out actuals",
    )
    ax.fill_between(
        x_forecast,
        y_q03[:, k],
        y_q97[:, k],
        color="orange",
        alpha=0.18,
        label="94% HDI",
    )
    ax.fill_between(
        x_forecast,
        y_q10[:, k],
        y_q90[:, k],
        color="orange",
        alpha=0.35,
        label="80% HDI",
    )
    ax.plot(
        x_forecast,
        y_median[:, k],
        color="darkorange",
        linewidth=2.0,
        label="Posterior median",
    )
    ax.axvline(T_TRAIN - 0.5, color="grey", linewidth=0.8, linestyle=":")
    ax.set_ylabel(var_labels[k], fontsize=12)
    ax.legend(loc="upper left", fontsize=9)
    ax.set_title(f"Variable {k} — {FORECAST_STEPS}-step forecast fan chart", fontsize=11)

axes[-1].set_xlabel("Time index", fontsize=11)
fig.suptitle(
    "BayesianVECM posterior predictive"
    f" — train t=0..{T_TRAIN - 1}, forecast t={T_TRAIN}..{T_TOTAL - 1}",
    fontsize=12,
    y=1.01,
)
plt.tight_layout()
plt.show()

## 6. Uncertainty propagation — watching the bands widen

The widening of the HDI bands is the direct consequence of compounding innovations. Let's measure the 80% band width at each step to make it concrete.

In [ ]:
band_width = y_q90 - y_q10  # (steps, K)

print("80% HDI band width by forecast step:")
print(f"{'step':>4}  {'y0 width':>10}  {'y1 width':>10}")
for h in range(FORECAST_STEPS):
    print(f"{h + 1:>4}  {band_width[h, 0]:>10.3f}  {band_width[h, 1]:>10.3f}")

print()
print("Width ratio (step 20 / step 1):")
for k in range(n_vars):
    ratio = band_width[-1, k] / band_width[0, k]
    print(f"  y{k}: {ratio:.2f}x wider at the horizon than at step 1")

## 7. Coverage check

With only 20 held-out points the coverage estimate is noisy, but it's worth a quick sanity check. For an 80% band we'd expect roughly 16 of the 20 actuals inside it.

In [ ]:
for k in range(n_vars):
    inside_80 = int(np.sum((endog_test[:, k] >= y_q10[:, k]) & (endog_test[:, k] <= y_q90[:, k])))
    inside_94 = int(np.sum((endog_test[:, k] >= y_q03[:, k]) & (endog_test[:, k] <= y_q97[:, k])))
    print(
        f"y{k}:  80% HDI coverage = {inside_80}/{T_TEST} ({100 * inside_80 // T_TEST}%)  "
        f" 94% HDI coverage = {inside_94}/{T_TEST} ({100 * inside_94 // T_TEST}%)"
    )

## 8. The error-correction term in the forecast

One of the distinctive features of a VECM forecast (vs a plain VAR-in-differences) is that the error-correction term keeps pulling both series back towards the cointegrating relation. We can see this by computing $\hat{\beta}^{\top} y_{T+h}$ along the posterior median path — it should stay close to zero rather than drifting.

This is what distinguishes VECM from a VAR-in-differences: a VAR-in-differences ignores the cointegrating relation entirely, so level forecasts can drift apart indefinitely. The VECM doesn't — the error-correction mechanism keeps their long-run combination anchored.

In [ ]:
# Posterior mean of beta[1, 0] (the free entry; beta[0, 0] = 1 always)
beta1_mean = float(model.idata_.posterior["beta"].values[:, :, 1, 0].mean())
print(f"Posterior mean beta[1, 0] = {beta1_mean:.4f}  (true = -0.5)")

ec_median_forecast = y_median[:, 0] + beta1_mean * y_median[:, 1]
ec_true_test = endog_test[:, 0] - 0.5 * endog_test[:, 1]
ec_true_train = endog_train[:, 0] - 0.5 * endog_train[:, 1]

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(
    x_train[-CONTEXT_ROWS:],
    ec_true_train[-CONTEXT_ROWS:],
    color="steelblue",
    label="Training  $\\hat{\\beta}^{\\top} y_t$",
)
ax.plot(x_forecast, ec_true_test, color="black", linestyle="--", label="Held-out actuals")
ax.plot(x_forecast, ec_median_forecast, color="darkorange", linewidth=2.0, label="Forecast median")
ax.axhline(0, color="grey", linewidth=0.8, linestyle=":")
ax.axvline(T_TRAIN - 0.5, color="grey", linewidth=0.8, linestyle=":")
ax.set_xlabel("Time index")
ax.set_ylabel("$\\hat{\\beta}^{\\top} y_t$")
ax.set_title("Error-correction term along the forecast path")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print(f"\nec_t in training       : mean={ec_true_train.mean():.3f}, std={ec_true_train.std():.3f}")
print(f"ec_t in held-out test  : mean={ec_true_test.mean():.3f}, std={ec_true_test.std():.3f}")
ec_fc_mean = ec_median_forecast.mean()
ec_fc_std = ec_median_forecast.std()
print(f"ec_t median forecast   : mean={ec_fc_mean:.3f}, std={ec_fc_std:.3f}")

## 9. What this unlocks — and what's still to come

**Shipped this slice (`feat/posterior-predictive`):**

The original target API from NOTES.md is now complete end-to-end:

```python
model = BayesianVECM(k_ar_diff=1, coint_rank=1, deterministic="n")
model.fit(endog_train)                                    # posterior
model.summary()                                           # tabular summary
pp = model.sample_posterior_predictive(steps=20)          # forecast distribution
y  = pp["posterior_predictive"]["y"].values               # (chain, draw, steps, K)
```

The forecasting loop is NumPy-only: fully vectorised over (chain x draw), single Python loop over steps, innovations drawn from the posterior Cholesky of $\Sigma$ pre-computed once. Returns an `xr.DataTree` with `posterior_predictive/y` (levels) and `posterior_predictive/delta_y` (differences).

**Still v0-scoped:**

- `fit` and `sample_posterior_predictive` still require `coint_rank=1` and `deterministic="n"`. The graph extension (higher ranks, deterministic terms) is the next estimator slice.
- No Bayesian model averaging across ranks — that waits until the wider graph is in.
- No diagnostics notebook — convergence checks and trace plots are worth their own walkthrough once we have more model variants to compare.